In [1]:
import os
api_key = os.getenv("OPENAI_API_KEY")
from dotenv import load_dotenv
load_dotenv()

from langchain_openai import ChatOpenAI

In [2]:
llm = ChatOpenAI(model="gpt-5-mini", temperature=0, api_key=api_key)

In [3]:
response = llm.invoke("Tell me 1 bad thing about Bangladesh!")
response.content

'Vulnerability to climate change and natural disasters — much of Bangladesh is low-lying delta, so it faces frequent cyclones, flooding, river erosion and increasing salinity and sea‑level threats, which displace people, damage crops and infrastructure, and strain the country’s development efforts.\n\nWould you like a different kind of issue (economic, social, governance, infrastructure) or more detail on this one?'

# RAG IMPLEMENTATION WITH TEXT DATA ONLY

In [4]:
from langchain_core.documents import Document

In [5]:
my_text = '''
Bangladesh,[b] officially the People's Republic of Bangladesh,[c] is a country in South Asia. It is the eighth-most populous country in the world with a population of almost 180 million within an area of 148,461 square kilometres (57,321 sq mi).[19][20] Bangladesh shares land borders with India to the north, west, and east, and Myanmar to the southeast. It has a coastline along the Bay of Bengal to its south and is separated from Bhutan and Nepal by the Siliguri Corridor, and from China by the Indian states of West Bengal and Sikkim to its north. Dhaka, the capital and largest city, is the nation's political, financial, and cultural centre. Chittagong is the second-largest city and the busiest port of the country.

The territory of present-day Bangladesh was ruled by successive Hindu and Buddhist dynasties in antiquity. Following the Muslim conquest in 1204, the region came under Bengal Sultanate and Mughal rule. As the largest subdivision of the Mughal Empire, Bengal became a major commercial centre, known for its textile industry and agricultural output. The Battle of Plassey in 1757 began British colonial rule, which lasted nearly two centuries; colonial policies redirected wealth from Bengal to Britain, while the region itself suffered repeated deadly famines. After the partition of India in 1947, East Bengal became the eastern wing of the newly formed Dominion of Pakistan and was later renamed East Pakistan.[21]

During more than two decades of political and economic marginalisation under the government based in West Pakistan, East Pakistan saw growing civil unrest that culminated in the 1948–52 language movement, 1969 mass uprising and the 1971 non-cooperation movement. Following the Pakistani military's Operation Searchlight, the Bangladesh Liberation War began. The Mukti Bahini, aided by Indian forces, fought a war of independence, during which Pakistani forces carried out the Bangladesh genocide; Bangladesh became a sovereign nation on 16 December 1971. After independence, Sheikh Mujibur Rahman led the country until his assassination in 1975; Ziaur Rahman, who later came to power, was himself assassinated in 1981. The 1980s were dominated by the dictatorship of Hussain Muhammad Ershad, who was overthrown in a mass uprising in 1990. Since the restoration to parliamentary democracy in 1991, national politics has largely dominated between the Awami League, led by Sheikh Hasina, and the Bangladesh Nationalist Party, led by Khaleda Zia. After more than 15 years of autocratic rule, Hasina was ousted in another mass uprising in 2024.[21]

Bangladesh is a unitary parliamentary republic based on the Westminster system. It has South Asia's second-largest economy and is one of the largest contributors to United Nations peacekeeping operations.[22] Bangladesh consists of eight divisions, 64 districts, 497 upazilas, and 4,599 union councils.[23] It is home to the largest mangrove forest in the world and hosts one of the world's largest refugee populations.[24] Bangladesh faces challenges in corruption, human rights concerns, political instability, and the effects of climate change. Bangladesh ranks one of the lowest in key social indicators: education, healthcare, housing, peacefulness, passport strength, personal safety, infrastructure and is one of the least developed countries according to UNCTAD.[25] Bangladesh is a member state of SAARC and other regional and international organisations.

'''

In [6]:
from langchain_core.documents import Document
docs = [Document(page_content=my_text, metadata={"id":"D01", "country":"Bangladesh"})]

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)
chunks = splitter.split_documents(docs)

In [8]:
chunks

[Document(metadata={'id': 'D01', 'country': 'Bangladesh'}, page_content="Bangladesh,[b] officially the People's Republic of Bangladesh,[c] is a country in South Asia. It is the eighth-most populous country in the world with a population of almost 180 million within an area of 148,461 square kilometres (57,321 sq mi).[19][20] Bangladesh shares land borders with India to the north, west, and east, and Myanmar to the southeast. It has a coastline along the Bay of Bengal to its south and is separated from Bhutan and Nepal by the Siliguri Corridor, and from China by the"),
 Document(metadata={'id': 'D01', 'country': 'Bangladesh'}, page_content="by the Siliguri Corridor, and from China by the Indian states of West Bengal and Sikkim to its north. Dhaka, the capital and largest city, is the nation's political, financial, and cultural centre. Chittagong is the second-largest city and the busiest port of the country."),
 Document(metadata={'id': 'D01', 'country': 'Bangladesh'}, page_content='The

In [9]:
from langchain_openai import OpenAIEmbeddings
embedding_model = OpenAIEmbeddings()

In [10]:
from langchain_community.vectorstores import Chroma 
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model
)

C:\Users\PIYAL\AppData\Local\Temp\ipykernel_15232\1271255505.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma


In [15]:
vectorstore.similarity_search("I need some reasons to prove that bangladesh is not a good country", k=3)


[Document(metadata={'id': 'D01', 'country': 'Bangladesh'}, page_content='in corruption, human rights concerns, political instability, and the effects of climate change. Bangladesh ranks one of the lowest in key social indicators: education, healthcare, housing, peacefulness, passport strength, personal safety, infrastructure and is one of the least developed countries according to UNCTAD.[25] Bangladesh is a member state of SAARC and other regional and international organisations.'),
 Document(metadata={'country': 'Bangladesh', 'id': 'D01'}, page_content="Bangladesh,[b] officially the People's Republic of Bangladesh,[c] is a country in South Asia. It is the eighth-most populous country in the world with a population of almost 180 million within an area of 148,461 square kilometres (57,321 sq mi).[19][20] Bangladesh shares land borders with India to the north, west, and east, and Myanmar to the southeast. It has a coastline along the Bay of Bengal to its south and is separated from Bhut

In [16]:
context = vectorstore.similarity_search("I need some reasons to prove that bangladesh is not a good country", k=3)


In [21]:
response = llm.invoke(f'First understand the following information: {context}. Now tell me some points why a person should like or dislike bangladesh.')
response.content

'Here are concise, balanced points someone might consider when deciding whether they would like or dislike Bangladesh. I grouped them so you can quickly see positives and negatives; individual experience will depend on personal priorities (e.g., nature, culture, stability, services).\n\nReasons someone might like Bangladesh\n- Rich culture and heritage: centuries of literature, music, festivals (Bengali culture is vibrant and expressive).\n- Warm hospitality: many visitors note friendly, welcoming people and strong community ties.\n- Natural attractions: Sundarbans (largest mangrove forest), long coastlines and beaches (e.g., Cox’s Bazar), and many rivers and wetlands.\n- Tasty and distinctive cuisine: flavorful rice- and fish-centered Bengali food, street food culture.\n- Low cost of living: travel, food and services are often inexpensive for many visitors/expats.\n- Rapid economic growth and opportunity: one of South Asia’s larger economies with a strong ready-made garments sector an

In [22]:
print(response.content)

Here are concise, balanced points someone might consider when deciding whether they would like or dislike Bangladesh. I grouped them so you can quickly see positives and negatives; individual experience will depend on personal priorities (e.g., nature, culture, stability, services).

Reasons someone might like Bangladesh
- Rich culture and heritage: centuries of literature, music, festivals (Bengali culture is vibrant and expressive).
- Warm hospitality: many visitors note friendly, welcoming people and strong community ties.
- Natural attractions: Sundarbans (largest mangrove forest), long coastlines and beaches (e.g., Cox’s Bazar), and many rivers and wetlands.
- Tasty and distinctive cuisine: flavorful rice- and fish-centered Bengali food, street food culture.
- Low cost of living: travel, food and services are often inexpensive for many visitors/expats.
- Rapid economic growth and opportunity: one of South Asia’s larger economies with a strong ready-made garments sector and growing